# Stage 1 & 2 — Multi-Dataset Data Input + Preprocessing (Binary: Cancer vs Normal)
### Cancer Prediction from Gene Expression Data — Prediction Track

This notebook combines **multiple breast cancer GEO datasets** into one larger, more robust training set
for **binary** classification (Tumor vs Normal), instead of relying on a single small dataset.

**Datasets used (all on the same GPL570 Affymetrix platform, so they share the same probe/gene IDs):**

| Accession | Approx. Samples |
|---|---|
| GSE10810 | 58 |
| GSE17907 | 55 |
| GSE20711 | 90 |
| GSE42568 | 121 (104 tumor + 17 normal) |
| GSE45827 | 155 (130 tumor + 11 normal, + some cell lines to exclude) |
| GSE61304 | 62 |

Combined, published work merging these six series reports roughly **476 tumor + 65 normal** samples after
cleaning — a much bigger and more robust dataset than any single series alone.

### Important — manual verification step required
GEO datasets don't use a single consistent metadata field for "is this tumor or normal tissue." You will
need to briefly inspect each dataset's sample metadata (this notebook prints it for you) and confirm/adjust
the keyword rules in the `LABEL_RULES` dictionary below before trusting the labels. This is a normal part
of working with real GEO data — there is no way to fully automate it reliably across independently
submitted datasets.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 0. Setup

In [3]:
!pip install GEOparse combat imbalanced-learn -q

  Preparing metadata (setup.py) ... done


In [4]:
import numpy as np
import pandas as pd
import GEOparse
from combat.pycombat import pycombat
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import joblib
import os

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUTPUT_DIR = "processed_data"
GEO_CACHE_DIR = "geo_cache"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(GEO_CACHE_DIR, exist_ok=True)

ACCESSIONS = ["GSE10810", "GSE17907", "GSE20711", "GSE42568", "GSE45827", "GSE61304"]

In [5]:
from google.colab import drive
drive.mount('/content/drive')

# Uncomment and point these to folders in your Drive, then re-run the cell above with these paths
# OUTPUT_DIR = "/content/drive/MyDrive/cancer-prediction-project/processed_data"
# GEO_CACHE_DIR = "/content/drive/MyDrive/cancer-prediction-project/geo_cache"
# os.makedirs(OUTPUT_DIR, exist_ok=True)
# os.makedirs(GEO_CACHE_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Download All GEO Datasets

In [6]:
gse_objects = {}

for acc in ACCESSIONS:
    print(f"Downloading {acc} ...")
    gse_objects[acc] = GEOparse.get_GEO(geo=acc, destdir=GEO_CACHE_DIR, silent=True)

print("All datasets downloaded.")

/usr/local/lib/python3.12/dist-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")


/usr/local/lib/python3.12/dist-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")


/usr/local/lib/python3.12/dist-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")


/usr/local/lib/python3.12/dist-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")


/usr/local/lib/python3.12/dist-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")


/usr/local/lib/python3.12/dist-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")


All datasets downloaded.


## 2. Inspect Sample Metadata (REQUIRED before trusting labels)
Run this and actually read the output for each dataset. You're looking for the metadata field (usually
`characteristics_ch1`, `source_name_ch1`, or `title`) that indicates tumor vs normal tissue, and the exact
wording used (e.g. "tumor", "cancer", "normal", "adjacent normal", "control").

In [7]:
for acc, gse in gse_objects.items():
    print(f"\n{'='*60}\n{acc}  ({len(gse.gsms)} samples)\n{'='*60}")
    # print metadata for the first 3 samples as a quick preview
    for i, (gsm_name, gsm) in enumerate(gse.gsms.items()):
        if i >= 3:
            break
        print(f"  Sample: {gsm_name}")
        print(f"    title: {gsm.metadata.get('title')}")
        print(f"    source_name_ch1: {gsm.metadata.get('source_name_ch1')}")
        print(f"    characteristics_ch1: {gsm.metadata.get('characteristics_ch1')}")


GSE10810  (58 samples)
  Sample: GSM272923
    title: ['Control Con_rep1']
    source_name_ch1: ['Human Breast']
    characteristics_ch1: ['t: S', 'n: 0', 'histology: 0', 'grade: 0', 'receptor: 0', 'ki-67: 0', 'tumor (t) vs healthy (s): S', 'phenotypes: S']
  Sample: GSM272924
    title: ['Control Con_rep2']
    source_name_ch1: ['Human Breast']
    characteristics_ch1: ['t: S', 'n: 0', 'histology: 0', 'grade: 0', 'receptor: 0', 'ki-67: 0', 'tumor (t) vs healthy (s): S', 'phenotypes: S']
  Sample: GSM272925
    title: ['Control Con_rep3']
    source_name_ch1: ['Human Breast']
    characteristics_ch1: ['t: S', 'n: 0', 'histology: 0', 'grade: 0', 'receptor: 0', 'ki-67: 0', 'tumor (t) vs healthy (s): S', 'phenotypes: S']

GSE17907  (109 samples)
  Sample: GSM447197
    title: ['BC5836']
    source_name_ch1: ['Breast cancer']
    characteristics_ch1: ['center: IPC', 'type: IBC', 'age of diagnosis (year): 36', 'molecular subtype: ERBB2', 'histology: DUC', 'pt: pT3', 'pn: --', 'grade sbr: 3

## 3. Label Rules
Fill in / adjust these based on what you saw above. For each dataset, specify:
- `field`: which metadata field to read (`"title"`, `"source_name_ch1"`, or `"characteristics_ch1"`)
- `normal_keywords`: substrings (lowercase) that indicate a NORMAL sample
- `exclude_keywords`: substrings that mean "drop this sample" (e.g. cell lines, which aren't real tissue)

Anything that matches neither `normal_keywords` nor `exclude_keywords` is labeled TUMOR by default —
this is safe here because these are all breast-cancer-focused series where the large majority of samples
are tumor tissue, but double check the "Unmatched / defaulted to tumor" printout below.

In [8]:
LABEL_RULES = {
    "GSE10810": {"field": "characteristics_ch1", "normal_keywords": ["normal"], "exclude_keywords": []},
    "GSE17907": {"field": "characteristics_ch1", "normal_keywords": ["normal"], "exclude_keywords": []},
    "GSE20711": {"field": "characteristics_ch1", "normal_keywords": ["normal"], "exclude_keywords": []},
    "GSE42568": {"field": "characteristics_ch1", "normal_keywords": ["normal"], "exclude_keywords": []},
    "GSE45827": {"field": "characteristics_ch1", "normal_keywords": ["normal"], "exclude_keywords": ["cell line"]},
    "GSE61304": {"field": "characteristics_ch1", "normal_keywords": ["normal"], "exclude_keywords": []},
}

def get_field_text(gsm, field):
    value = gsm.metadata.get(field, [])
    return " ".join(value).lower() if isinstance(value, list) else str(value).lower()

def label_sample(gsm, rules):
    text = get_field_text(gsm, rules["field"])
    if any(kw in text for kw in rules["exclude_keywords"]):
        return "exclude"
    if any(kw in text for kw in rules["normal_keywords"]):
        return "normal"
    return "tumor"   # default assumption for these breast-cancer-focused series

## 4. Extract Expression Data + Labels, Align Genes Across Datasets

In [9]:
dataset_frames = []   # list of (expression_df [genes x samples], label_series, batch_id)
unmatched_log = []

for acc, gse in gse_objects.items():
    rules = LABEL_RULES[acc]
    expr = gse.pivot_samples("VALUE")   # genes (rows) x samples (columns)

    labels = {}
    for gsm_name, gsm in gse.gsms.items():
        lbl = label_sample(gsm, rules)
        if lbl == "exclude":
            continue
        labels[gsm_name] = lbl
        text = get_field_text(gsm, rules["field"])
        if lbl == "tumor" and not any(kw in text for kw in ["tumor", "cancer", "carcinoma", "invasive", "malignant"]):
            unmatched_log.append((acc, gsm_name, text))

    keep_samples = list(labels.keys())
    expr = expr[keep_samples]

    dataset_frames.append((acc, expr, pd.Series(labels)))
    print(f"{acc}: kept {len(keep_samples)} samples "
          f"({sum(v=='tumor' for v in labels.values())} tumor, "
          f"{sum(v=='normal' for v in labels.values())} normal)")

print(f"\nSamples defaulted to 'tumor' without an explicit tumor/cancer keyword match "
      f"({len(unmatched_log)} total) — spot-check a few of these:")
for acc, gsm_name, text in unmatched_log[:10]:
    print(f"  [{acc}] {gsm_name}: {text}")

GSE10810: kept 58 samples (58 tumor, 0 normal)
GSE17907: kept 109 samples (101 tumor, 8 normal)
GSE20711: kept 90 samples (88 tumor, 2 normal)
GSE42568: kept 121 samples (104 tumor, 17 normal)
GSE45827: kept 141 samples (130 tumor, 11 normal)
GSE61304: kept 62 samples (58 tumor, 4 normal)

Samples defaulted to 'tumor' without an explicit tumor/cancer keyword match (189 total) — spot-check a few of these:
  [GSE17907] GSM447197: center: ipc type: ibc age of diagnosis (year): 36 molecular subtype: erbb2 histology: duc pt: pt3 pn: -- grade sbr: 3 erbb2 ihc status: pos er ihc status: neg pr ihc status: neg p53 ihc status: neg ki67 ihc status: pos erbb2.ht ihc status: pos erbb2.tab ihc status: pos erbb2p ihc status: pos igf1r ihc status: neg egfr ihc status: pos top2a ihc status: pos foxa1 ihc status: pos mfs: 1 mfsdel (month): 13.14
  [GSE17907] GSM447198: center: ipc type: ibc age of diagnosis (year): 68 molecular subtype: erbb2 histology: duc pt: -- pn: -- grade sbr: 3 erbb2 ihc status: 

In [10]:
# Align on genes common to ALL datasets (should be near-total overlap since all are GPL570,
# but do this properly rather than assuming)
common_genes = set(dataset_frames[0][1].index)
for _, expr, _ in dataset_frames[1:]:
    common_genes &= set(expr.index)
common_genes = sorted(common_genes)

print(f"Common genes across all {len(dataset_frames)} datasets: {len(common_genes)}")

aligned_expr = []
aligned_labels = []
aligned_batch = []

for acc, expr, labels in dataset_frames:
    expr_aligned = expr.loc[common_genes]
    aligned_expr.append(expr_aligned)
    aligned_labels.append(labels)
    aligned_batch.extend([acc] * expr_aligned.shape[1])

# Combine into one big genes x samples matrix
combined_expr = pd.concat(aligned_expr, axis=1)          # genes x samples
combined_labels = pd.concat(aligned_labels)               # sample -> tumor/normal
combined_batch = pd.Series(aligned_batch, index=combined_expr.columns)

print("Combined expression matrix (genes x samples):", combined_expr.shape)
print("Combined label counts:", combined_labels.value_counts().to_dict())
print("Samples per batch (dataset):", combined_batch.value_counts().to_dict())

Common genes across all 6 datasets: 16609
Combined expression matrix (genes x samples): (16609, 581)
Combined label counts: {'tumor': 539, 'normal': 42}
Samples per batch (dataset): {'GSE45827': 141, 'GSE42568': 121, 'GSE17907': 109, 'GSE20711': 90, 'GSE61304': 62, 'GSE10810': 58}


## 5. Log2 Normalization (if needed)
Microarray data processed with RMA (common for GPL570 series) is usually **already log2-transformed**.
Re-applying log2 on top of already-logged data would be wrong. This cell auto-detects the scale and only
applies log2 if the data still looks like raw (linear-scale) intensities.

In [11]:
max_val = combined_expr.values.max()
print("Max value in combined data:", max_val)

if max_val > 50:
    print("Data looks like raw/linear-scale intensities -> applying log2(x + 1)")
    combined_expr = np.log2(combined_expr.clip(lower=0) + 1)
else:
    print("Data already appears log-transformed -> skipping log2 step")

print("New max value:", combined_expr.values.max())

Max value in combined data: nan
Data already appears log-transformed -> skipping log2 step
New max value: nan


## 6. Missing Value Handling

In [12]:
print("Total missing values:", combined_expr.isna().sum().sum())

# Fill missing readings with that gene's mean expression across all samples
combined_expr = combined_expr.apply(lambda row: row.fillna(row.mean()), axis=1)

print("Missing values after imputation:", combined_expr.isna().sum().sum())

Total missing values: 896886
Missing values after imputation: 0


## 7. Batch Effect Correction (ComBat)


In [13]:
batch_list = combined_batch.loc[combined_expr.columns].tolist()

combined_expr_corrected = pycombat(combined_expr, batch_list)

print("Batch-corrected expression matrix shape:", combined_expr_corrected.shape)

Found 6 batches.
Adjusting for 0 covariate(s) or covariate level(s).
Standardizing Data across genes.
Fitting L/S model and finding priors.
Finding parametric adjustments.
Adjusting the Data
Batch-corrected expression matrix shape: (16609, 581)


## 8. Transpose to Samples x Genes and Build Final Labels


In [14]:
X = combined_expr_corrected.T   # samples x genes
y_text = combined_labels.loc[X.index]
y = (y_text == "tumor").astype(int).values   # 1 = tumor/cancer, 0 = normal

print("X shape (samples x genes):", X.shape)
print("Class balance:", pd.Series(y).value_counts().to_dict(), " (1 = tumor, 0 = normal)")
joblib.dump(list(X.columns), f"{OUTPUT_DIR}/all_input_genes.pkl")

X shape (samples x genes): (581, 16609)
Class balance: {1: 539, 0: 42}  (1 = tumor, 0 = normal)


['processed_data/all_input_genes.pkl']

## 9. Low-Variance Gene Removal

In [15]:
var_selector = VarianceThreshold(threshold=0.05)
X_var = var_selector.fit_transform(X)

kept_genes = X.columns[var_selector.get_support()]
X_var = pd.DataFrame(X_var, columns=kept_genes, index=X.index)

print(f"Genes before: {X.shape[1]}  ->  Genes after low-variance removal: {X_var.shape[1]}")

Genes before: 16609  ->  Genes after low-variance removal: 16488


## 10. Train / Test Split
Split before scaling and SMOTE to avoid data leakage — same reasoning as the single-dataset version.

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X_var, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)
print("Train class counts:", pd.Series(y_train).value_counts().to_dict())
print("Test class counts:", pd.Series(y_test).value_counts().to_dict())

Train shape: (464, 16488)  Test shape: (117, 16488)
Train class counts: {1: 430, 0: 34}
Test class counts: {1: 109, 0: 8}


## 11. Z-score Scaling (fit on train only)

In [17]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print("Train scaled shape:", X_train_scaled.shape)

Train scaled shape: (464, 16488)


## 12. SMOTE — Fix Class Imbalance


In [18]:
print("Before SMOTE:", pd.Series(y_train).value_counts().to_dict())

smote = SMOTE(random_state=RANDOM_STATE)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print("After SMOTE: ", pd.Series(y_train_balanced).value_counts().to_dict())
print("Balanced train shape:", X_train_balanced.shape)

Before SMOTE: {1: 430, 0: 34}
After SMOTE:  {1: 430, 0: 430}
Balanced train shape: (860, 16488)


## 13. Save Processed Data


In [19]:
X_train_balanced.to_csv(f"{OUTPUT_DIR}/X_train.csv", index=False)
X_test_scaled.to_csv(f"{OUTPUT_DIR}/X_test.csv", index=False)
pd.Series(y_train_balanced).to_csv(f"{OUTPUT_DIR}/y_train.csv", index=False)
pd.Series(y_test).to_csv(f"{OUTPUT_DIR}/y_test.csv", index=False)

joblib.dump(scaler, f"{OUTPUT_DIR}/scaler.pkl")
joblib.dump(var_selector, f"{OUTPUT_DIR}/variance_selector.pkl")
joblib.dump(list(X_train_balanced.columns), f"{OUTPUT_DIR}/genes_after_variance_filter.pkl")

# Save the label mapping so everyone on the team uses the same convention
joblib.dump({0: "normal", 1: "tumor"}, f"{OUTPUT_DIR}/label_mapping.pkl")

print("Saved all preprocessing outputs to:", OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))

Saved all preprocessing outputs to: processed_data
['X_train.csv', 'y_train.csv', 'scaler.pkl', 'X_test.csv', 'y_test.csv', 'genes_after_variance_filter.pkl', 'all_input_genes.pkl', 'label_mapping.pkl', 'variance_selector.pkl']
